# Optogenetic control of a neuron: Hodgkin-Huxley + channelrhodopsin-2

## Problem
Optogenetics makes neurons fire when illuminated. How much light, for how long, and at what rate can we stimulate before the control breaks down, and can the unknown light sensitivity of a cell be recovered from its spikes?

## Background
Hodgkin and Huxley (Nobel Prize in Physiology or Medicine, 1963) described the action potential with voltage-gated sodium and potassium channels. Channelrhodopsin-2 (ChR2), used by Boyden, Deisseroth and colleagues (Breakthrough Prize 2013, Lasker Award 2021), adds a light-gated cation channel so light becomes an input current.

## Data
This is a simulation, not a measurement: standard Hodgkin-Huxley squid-axon parameters and a simplified four-state ChR2 photocycle (closed C1, open O1 and O2, closed C2). The kinetic constants are illustrative and are not fitted to experimental recordings.

## Method
$$C_m\frac{dV}{dt} = -g_{Na}m^3h(V-E_{Na}) - g_Kn^4(V-E_K) - g_L(V-E_L) - g_{ChR2}(O_1+\gamma O_2)(V-E_{ChR2}),\qquad \frac{dx}{dt}=\alpha_x(V)(1-x)-\beta_x(V)x$$
The ChR2 states follow a Markov chain whose activation rates scale with the light input $\phi$. The stiff system is integrated with LSODA. Light sensitivity is estimated with approximate Bayesian computation (ABC): sample parameters from a prior, simulate, and keep the 10% closest to the observed spike count and latency.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, matplotlib.pyplot as plt
import optogenetics as O
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
pulses = [(20, 25, 1.0), (80, 85, 1.0)]
sol = O.simulate(pulses, T=150)
fig, ax = plt.subplots(3, 1, figsize=(7.5, 6.5), sharex=True)
ax[0].plot(sol.t, sol.y[0], lw=1); ax[0].set_ylabel('V (mV)'); ax[0].set_title('Light-evoked action potentials')
C1, O1, O2, C2 = sol.y[4:8]
ax[1].plot(sol.t, C1, label='C1'); ax[1].plot(sol.t, O1, label='O1'); ax[1].plot(sol.t, O2, label='O2'); ax[1].plot(sol.t, C2, label='C2'); ax[1].legend(ncol=4, fontsize=8); ax[1].set_ylabel('ChR2 state')
ax[2].plot(sol.t, [O.light(t, pulses) for t in sol.t]); ax[2].set_ylabel('light'); ax[2].set_xlabel('time (ms)')
plt.tight_layout(); plt.show()
print('Spike times (ms):', np.round(O.spike_times(sol), 1))

Spike times (ms): [22.  82.4]


In [2]:
# Control experiment: same light, but no opsin (g_ChR2 = 0)
ctl = O.simulate(pulses, T=150, g_chr2=0.0)
print('Spikes with ChR2:', len(O.spike_times(sol)), '| without ChR2:', len(O.spike_times(ctl)))
plt.figure(figsize=(7,2.8)); plt.plot(sol.t, sol.y[0], label='with ChR2'); plt.plot(ctl.t, ctl.y[0], label='no opsin'); plt.legend(); plt.xlabel('ms'); plt.ylabel('V (mV)'); plt.show()

Spikes with ChR2: 2 | without ChR2: 0


In [3]:
# Strength-duration: minimum irradiance that triggers a spike for each pulse width
widths = [1, 2, 3, 5, 8, 12]
grid = np.linspace(0.02, 1.5, 38)
thr = []
for w in widths:
    t = np.nan
    for I in grid:
        if len(O.spike_times(O.simulate([(10, 10+w, I)], T=60))) > 0: t = I; break
    thr.append(t)
print(dict(zip(widths, np.round(thr, 3))))
plt.figure(figsize=(5,3.4)); plt.plot(widths, thr, 'o-'); plt.xlabel('pulse width (ms)'); plt.ylabel('threshold irradiance (a.u.)'); plt.title('Strength-duration curve'); plt.show()

{1: np.float64(0.18), 2: np.float64(0.1), 3: np.float64(0.06), 5: np.float64(0.06), 8: np.float64(0.06), 12: np.float64(0.06)}


In [4]:
# Dose-response: spikes per 100 ms train versus irradiance, for low/medium/high opsin expression
fig, ax = plt.subplots(figsize=(5.5,3.6))
irr = np.linspace(0.05, 1.5, 12)
for g in (0.2, 0.4, 0.8):
    n = [len(O.spike_times(O.simulate([(10, 100, I)], T=110, g_chr2=g))) for I in irr]
    print(f'g_ChR2={g}: spikes at irradiance {irr.round(2).tolist()} -> {n}')
    ax.plot(irr, n, 'o-', label=f'g_ChR2={g}')
ax.set_xlabel('irradiance (a.u.)'); ax.set_ylabel('spikes in 90 ms of continuous light'); ax.legend(); plt.show()

g_ChR2=0.2: spikes at irradiance [0.05, 0.18, 0.31, 0.45, 0.58, 0.71, 0.84, 0.97, 1.1, 1.24, 1.37, 1.5] -> [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


g_ChR2=0.4: spikes at irradiance [0.05, 0.18, 0.31, 0.45, 0.58, 0.71, 0.84, 0.97, 1.1, 1.24, 1.37, 1.5] -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


g_ChR2=0.8: spikes at irradiance [0.05, 0.18, 0.31, 0.45, 0.58, 0.71, 0.84, 0.97, 1.1, 1.24, 1.37, 1.5] -> [3, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]


In [5]:
fid = O.fidelity_sweep(freqs=(5, 10, 20, 40, 60, 80, 100), width=3.0, T=400)
for f, x in fid: print(f'{f:>4} Hz  fidelity {x:.2f}')
plt.figure(figsize=(5,3.4)); plt.plot(*zip(*fid), 'o-'); plt.xlabel('pulse frequency (Hz)'); plt.ylabel('spike fidelity'); plt.ylim(0,1.05); plt.title('Can the neuron follow the light?'); plt.show()

   5 Hz  fidelity 1.00
  10 Hz  fidelity 1.00
  20 Hz  fidelity 1.00
  40 Hz  fidelity 1.00
  60 Hz  fidelity 1.00
  80 Hz  fidelity 0.58
 100 Hz  fidelity 0.51


In [6]:
# Why fidelity drops: ChR2 population available at the start of each pulse during an 80 Hz train
T=400; period=1000/80
train = [(10+k*period, 13+k*period, 1.0) for k in range(int((T-10)/period))]
s = O.simulate(train, T=T)
fig, ax = plt.subplots(2,1, figsize=(7.5,4.5), sharex=True)
ax[0].plot(s.t, s.y[0], lw=.8); ax[0].set_ylabel('V (mV)')
ax[1].plot(s.t, s.y[4], label='C1 (ready to open)'); ax[1].plot(s.t, s.y[5]+s.y[6], label='open'); ax[1].legend(); ax[1].set_xlabel('ms')
plt.tight_layout(); plt.show()
print('Mean C1 fraction just before each pulse (late train):', np.round([s.y[4][np.searchsorted(s.t, a-0.5)] for a,_,_ in train[-4:]], 2))

Mean C1 fraction just before each pulse (late train): [0.19 0.19 0.19 0.19]


In [7]:
prior, post = O.infer_sensitivity(true_sens=0.7, n_draws=300, seed=0)
print(f'ABC posterior for light sensitivity (true 0.70): mean {post.mean():.2f}, sd {post.std():.2f}, 90% interval [{np.quantile(post,.05):.2f}, {np.quantile(post,.95):.2f}]')
plt.figure(figsize=(5,3.4)); plt.hist(prior, bins=20, alpha=.35, label='prior'); plt.hist(post, bins=10, alpha=.9, label='ABC posterior'); plt.axvline(0.7, c='r', ls='--', label='true'); plt.legend(); plt.xlabel('light sensitivity'); plt.show()

ABC posterior for light sensitivity (true 0.70): mean 0.69, sd 0.04, 90% interval [0.64, 0.77]


In [8]:
# Recovery across several true values: does ABC track the truth?
rows=[]
for ts in (0.3, 0.5, 0.7, 1.0):
    _, p = O.infer_sensitivity(true_sens=ts, n_draws=200, seed=1)
    rows.append((ts, p.mean(), p.std()))
for ts, m, sd in rows: print(f'true {ts:.2f} -> posterior {m:.2f} +/- {sd:.2f}')
plt.figure(figsize=(4,4)); plt.errorbar([r[0] for r in rows], [r[1] for r in rows], yerr=[r[2] for r in rows], fmt='o'); plt.plot([0.2,1.1],[0.2,1.1],'k--'); plt.xlabel('true sensitivity'); plt.ylabel('ABC estimate'); plt.show()

true 0.30 -> posterior 0.32 +/- 0.04
true 0.50 -> posterior 0.50 +/- 0.03
true 0.70 -> posterior 0.69 +/- 0.05
true 1.00 -> posterior 1.09 +/- 0.06


## Results
See the outputs above: the light-evoked spike train and ChR2 state occupancy, the no-opsin control, the strength-duration curve, the dose-response by expression level, the frequency fidelity sweep, the explanation of why fast trains fail, and the ABC recovery of the light sensitivity.

## Conclusions
1. **Light alone is enough to control spiking, and the opsin is the cause.** Two 5 ms pulses produce two spikes (at 22.0 ms and 82.4 ms, about 2 ms after light onset); the same light on a neuron without ChR2 produces none.
2. **Activation has a threshold that depends on pulse width.** The minimum irradiance that triggers a spike falls from 0.18 for a 1 ms pulse to 0.10 at 2 ms and 0.06 from 3 ms onward (the grid step is 0.04, so 0.06 is a resolution floor). This is the classic strength-duration trade-off: brief pulses need more light, long pulses only need to cross rheobase.
3. **More light does not mean more spikes; expression level does.** With 90 ms of continuous light, a cell with low expression (g_ChR2 = 0.2 or 0.4) fires a single spike at any irradiance above threshold, while a high-expression cell (0.8) fires up to 7 spikes and saturates from irradiance 0.31 upward. Beyond the threshold, the number of opsin channels sets the response, not the light intensity.
4. **Fast trains break the control.** Spike fidelity is 1.00 up to 60 Hz and falls to 0.58 at 80 Hz and 0.51 at 100 Hz. The simulated photocycle explains it: only about 19% of the channel population is back in the ready state (C1) just before each late pulse of an 80 Hz train, so later pulses cannot depolarise the cell enough. This matches the practical need for faster opsin variants in high-frequency experiments.
5. **The light sensitivity can be recovered from the spikes.** ABC returns 0.69 ± 0.04 (90% interval 0.64 to 0.77) for a true value of 0.70, and tracks other true values: 0.30 gives 0.32, 0.50 gives 0.50 and 1.00 gives 1.09, a slight overestimate at the top of the range.
6. **Limits.** Single-compartment neuron, illustrative kinetics not fitted to recordings, no light scattering in tissue, no temperature dependence and no noise. Absolute numbers (thresholds, frequencies) should not be compared with experiments; the qualitative mechanisms should.

**Takeaway for the interview:** the project is a complete simulate, probe and infer workflow: a mechanistic model that explains where control fails (channel recovery) and a likelihood-free method to estimate a parameter that cannot be measured directly.